# بازی ماز با الگوریتم پیمایش عمقی (DFS)

**طراح:** sadegh azaddavar  
**استاد:** mohammad hasan iranshahi


در این تمرین دو کار را با **DFS** انجام می‌دهیم:

1. **ساخت ماز** (Maze Generation): با روش Recursive Backtracker که همان DFS است.
2. **حل ماز** (Maze Solving): پیدا کردن مسیر از ورودی (بالا چپ) تا خروجی (پایین راست) با DFS.

## ایده‌ی DFS
از یک خانه شروع می‌کنیم و تا جای ممکن در یک مسیر **عمیق** می‌رویم. وقتی به بن‌بست رسیدیم، به خانه‌ی قبلی برمی‌گردیم (backtrack) و مسیر دیگری را امتحان می‌کنیم. برای این کار از یک **پشته (Stack)** استفاده می‌کنیم.

## نحوه‌ی نمایش ماز
ماز یک جدول `ROWS × COLS` است. برای هر خانه یک مجموعه نگه می‌داریم که نشان می‌دهد از آن خانه به کدام همسایه‌ها راه باز است (بدون دیوار). این ساختار در واقع یک **گراف** است: هر خانه یک گره و هر راه باز یک یال.

## گام ۰: آماده‌سازی محیط (رفع خطاهای NameError و ModuleNotFoundError)

اگر هنگام اجرا خطاهایی مثل `No module named 'matplotlib'` یا `name 'plt' is not defined` یا `name 'solve_dfs' is not defined` دیدید، علت‌ها این دو مورد است:

1. **کتابخانه‌ی matplotlib روی کِرنل (Kernel) انتخاب‌شده نصب نیست.** سلول بعدی آن را خودکار روی همان پایتونی که نوت‌بوک با آن اجرا می‌شود نصب می‌کند.
2. **سلول‌ها به ترتیب اجرا نشده‌اند.** مثلاً سلول تعریف `solve_dfs` اجرا نشده و سراغ سلول‌های بعدی رفته‌اید. هر `NameError` یعنی «سلولی که این نام را تعریف می‌کند هنوز اجرا نشده».

**راه‌حل:** اول سلول نصب را اجرا کنید، سپس از نوار بالا روی **Run All** بزنید تا همه‌ی سلول‌ها از بالا به پایین اجرا شوند. اگر بعد از نصب هنوز ModuleNotFoundError دیدید، یک‌بار **Restart** بزنید و دوباره Run All کنید. نمودار بعدی علت و راه‌حل را گرافیکی نشان می‌دهد.

In [ ]:
import sys
import subprocess
import importlib
import importlib.util

# module name : package name to install
REQUIRED = {'matplotlib': 'matplotlib', 'IPython': 'ipython'}

# pip variants, tried in order. Homebrew / system Pythons (macOS, Linux) often
# refuse a plain install with "externally-managed-environment", so we also
# try --user and --break-system-packages.
VARIANTS = [
    [],
    ['--user'],
    ['--break-system-packages'],
    ['--user', '--break-system-packages'],
]


def pip_install(package):
    for extra in VARIANTS:
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + extra + [package]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return True
        if 'No module named pip' in result.stderr:
            subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'])
    print(result.stderr[-600:])
    return False


all_ok = True
for module, package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print('Installing ' + package + ' ...')
        if not pip_install(package):
            all_ok = False
    else:
        print(module + ' is already installed')

importlib.invalidate_caches()

if importlib.util.find_spec('matplotlib') is not None:
    import matplotlib
    print('OK  matplotlib', matplotlib.__version__)
    print('Now press Restart (top bar), then Run All.')
else:
    print('Automatic install failed. Run this line in your terminal, then Restart + Run All:')
    print('    ' + sys.executable + ' -m pip install matplotlib ipython --break-system-packages')
print('python used here:', sys.executable)

### نمودار علت خطاها و راه‌حل

هر ردیف قرمز یک علت و خطای ناشی از آن را نشان می‌دهد و ستون سبز راه‌حل همان ردیف است.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 6.5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 7)
ax.axis('off')


def box(x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                                fc=color, ec='#111827', lw=1.6))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=10)


def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                                 mutation_scale=20, lw=2, color='#374151'))


# column titles
ax.text(2.0, 6.6, 'CAUSE', ha='center', fontsize=13, fontweight='bold')
ax.text(6.5, 6.6, 'ERROR YOU SEE', ha='center', fontsize=13, fontweight='bold')
ax.text(11.0, 6.6, 'FIX', ha='center', fontsize=13, fontweight='bold')

RED, GREEN = '#fecaca', '#bbf7d0'

# row 1: matplotlib is missing
box(0.3, 4.3, 3.4, 1.5, 'matplotlib is not installed\nin the selected kernel', RED)
box(4.8, 4.3, 3.4, 1.5, 'ModuleNotFoundError\nthen: name plt is not defined', RED)
box(9.3, 4.3, 3.4, 1.5, 'Run the setup cell\n(auto pip install)', GREEN)
arrow(3.8, 5.05, 4.7, 5.05)
arrow(8.3, 5.05, 9.2, 5.05)

# row 2: cells were not run in order
box(0.3, 2.0, 3.4, 1.5, 'The cell that defines\nsolve_dfs was not run', RED)
box(4.8, 2.0, 3.4, 1.5, 'NameError: solve_dfs\nthen: order, path not defined', RED)
box(9.3, 2.0, 3.4, 1.5, 'Press Run All\n(top to bottom order)', GREEN)
arrow(3.8, 2.75, 4.7, 2.75)
arrow(8.3, 2.75, 9.2, 2.75)

# result
box(4.3, 0.1, 4.4, 1.0, 'All cells run: maze + DFS solution', '#bfdbfe')
arrow(11.0, 1.9, 8.8, 0.8)

ax.set_title('Why the errors happen and how to fix them', fontsize=14)
plt.show()

## گام ۱: وارد کردن کتابخانه‌ها

In [ ]:
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# اجازه می‌دهیم انیمیشن در نوت‌بوک جا شود
plt.rcParams['animation.embed_limit'] = 50

## گام ۲: ساخت ماز با DFS

الگوریتم:
1. از خانه‌ی `(0, 0)` شروع کن و آن را علامت بزن و در پشته بگذار.
2. سر پشته را نگاه کن. اگر همسایه‌ی ملاقات‌نشده دارد، یکی را تصادفی انتخاب کن، دیوار بینشان را بردار و آن همسایه را در پشته بگذار.
3. اگر همسایه‌ی ملاقات‌نشده ندارد، از پشته یک خانه بردار (backtrack).
4. تا خالی شدن پشته ادامه بده.

چون هر خانه دقیقاً یک بار وارد ماز می‌شود، ماز نتیجه **کامل** است: بین هر دو خانه دقیقاً یک مسیر وجود دارد (بدون حلقه و بدون خانه‌ی غیرقابل دسترس).

In [ ]:
# چهار جهت حرکت: بالا، پایین، چپ، راست  (تغییر سطر، تغییر ستون)
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def generate_maze(rows, cols, seed=None):
    rnd = random.Random(seed)

    # در ابتدا همه‌ی دیوارها هستند، یعنی هیچ خانه‌ای راه بازی ندارد
    passages = {(r, c): set() for r in range(rows) for c in range(cols)}

    start = (0, 0)
    visited = {start}
    stack = [start]

    while stack:
        r, c = stack[-1]  # خانه‌ی فعلی (سر پشته)

        # همسایه‌های داخل جدول که هنوز ملاقات نشده‌اند
        neighbors = []
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                neighbors.append((nr, nc))

        if neighbors:
            nxt = rnd.choice(neighbors)
            # برداشتن دیوار بین دو خانه (راه دوطرفه)
            passages[(r, c)].add(nxt)
            passages[nxt].add((r, c))
            visited.add(nxt)
            stack.append(nxt)   # رفتن به عمق
        else:
            stack.pop()         # بن‌بست: برگشت

    return passages

## گام ۳: رسم ماز

تابع `draw_maze` ماز را با `matplotlib` می‌کشد:
- دیوارها: خطوط مشکی
- ورودی: خانه‌ی سبز (بالا چپ)
- خروجی: خانه‌ی زرد (پایین راست)
- خانه‌های بررسی‌شده توسط DFS: آبی روشن
- مسیر نهایی: قرمز

(عنوان نمودارها را انگلیسی نوشته‌ایم چون matplotlib متن فارسی را درست نمایش نمی‌دهد.)

In [ ]:
def draw_maze(passages, rows, cols, visited=(), path=(), current=None,
              title='', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6 * rows / cols))

    start, goal = (0, 0), (rows - 1, cols - 1)

    # رنگ کردن خانه‌ها (محور y را منفی می‌گیریم تا سطر ۰ بالا باشد)
    def paint(cell, color):
        r, c = cell
        ax.add_patch(Rectangle((c, -r - 1), 1, 1, color=color))

    for cell in visited:
        paint(cell, '#bfdbfe')
    for cell in path:
        paint(cell, '#fca5a5')
    paint(start, '#86efac')
    paint(goal, '#fde047')

    # موقعیت فعلی DFS
    if current is not None:
        ax.plot(current[1] + 0.5, -current[0] - 0.5, 'o',
                color='#1d4ed8', markersize=9)

    # جمع کردن دیوارها به صورت پاره‌خط
    walls = []
    for (r, c), opens in passages.items():
        if (r - 1, c) not in opens and (r, c) != start:   # دیوار بالا (ورودی باز است)
            walls.append([(c, -r), (c + 1, -r)])
        if (r + 1, c) not in opens and (r, c) != goal:    # دیوار پایین (خروجی باز است)
            walls.append([(c, -r - 1), (c + 1, -r - 1)])
        if (r, c - 1) not in opens:                       # دیوار چپ
            walls.append([(c, -r), (c, -r - 1)])
        if (r, c + 1) not in opens:                       # دیوار راست
            walls.append([(c + 1, -r), (c + 1, -r - 1)])
    ax.add_collection(LineCollection(walls, colors='black', linewidths=2))

    ax.set_xlim(-0.1, cols + 0.1)
    ax.set_ylim(-rows - 0.1, 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title)
    return ax

In [ ]:
ROWS, COLS = 15, 15

maze = generate_maze(ROWS, COLS, seed=42)   # seed ثابت یعنی هر بار همان ماز

draw_maze(maze, ROWS, COLS, title='Maze generated with DFS')
plt.show()

## گام ۴: حل ماز با DFS

الگوریتم:
1. خانه‌ی شروع را در پشته بگذار.
2. یک خانه از پشته بردار. اگر قبلاً دیده شده، رد شو.
3. آن را ملاقات‌شده علامت بزن. اگر خروجی بود، تمام.
4. همسایه‌های **قابل دسترس** (بدون دیوار) و ملاقات‌نشده را در پشته بگذار و برای هرکدام ثبت کن از کجا آمده‌ایم (`parent`).
5. در پایان با دنبال کردن `parent` از خروجی تا ورودی، مسیر را بازسازی می‌کنیم.

In [ ]:
def solve_dfs(passages, start, goal):
    stack = [start]
    parent = {start: None}   # هر خانه از کدام خانه آمده است
    visited = set()
    order = []               # ترتیب ملاقات خانه‌ها (برای نمایش)

    while stack:
        cell = stack.pop()
        if cell in visited:
            continue
        visited.add(cell)
        order.append(cell)

        if cell == goal:     # به خروجی رسیدیم
            break

        for nb in passages[cell]:
            if nb not in visited:
                parent[nb] = cell
                stack.append(nb)

    # بازسازی مسیر از خروجی به ورودی
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = parent.get(cell)
    path.reverse()

    return order, path

In [ ]:
start, goal = (0, 0), (ROWS - 1, COLS - 1)
order, path = solve_dfs(maze, start, goal)

print('تعداد کل خانه‌ها            :', ROWS * COLS)
print('تعداد خانه‌های بررسی‌شده   :', len(order))
print('طول مسیر نهایی              :', len(path))

draw_maze(maze, ROWS, COLS, visited=order, path=path,
          title='DFS solution (blue: explored, red: final path)')
plt.show()

## گام ۵: مراحل اجرای DFS به صورت تصویر

چهار مرحله از جستجو را کنار هم می‌بینیم تا رفتار «رفتن به عمق و برگشت از بن‌بست» مشخص شود.

In [ ]:
fractions = [0.1, 0.4, 0.7, 1.0]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, f in zip(axes, fractions):
    k = max(1, int(len(order) * f))
    final = (k == len(order))
    draw_maze(maze, ROWS, COLS,
              visited=order[:k],
              path=path if final else (),
              current=order[k - 1],
              title='Step ' + str(k) + ' of ' + str(len(order)),
              ax=ax)

plt.tight_layout()
plt.show()

## گام ۶: انیمیشن حل ماز

نقطه‌ی آبی پررنگ، خانه‌ی فعلی DFS است. در پایان مسیر درست با رنگ قرمز نشان داده می‌شود. روی دکمه‌ی Play زیر انیمیشن بزنید.

In [ ]:
def animate_solution(passages, rows, cols, order, path, max_frames=80):
    fig, ax = plt.subplots(figsize=(5, 5 * rows / cols))

    # برای ماز بزرگ، چند خانه را در هر فریم نشان می‌دهیم
    step = max(1, len(order) // max_frames)
    ks = list(range(step, len(order), step)) + [len(order)]
    total_frames = len(ks) + 10   # ۱۰ فریم آخر برای نمایش مسیر نهایی

    def update(i):
        ax.clear()
        if i < len(ks):
            k = ks[i]
            draw_maze(passages, rows, cols, visited=order[:k],
                      current=order[k - 1],
                      title='DFS exploring: ' + str(k) + ' / ' + str(len(order)),
                      ax=ax)
        else:
            draw_maze(passages, rows, cols, visited=order, path=path,
                      title='Path found! length = ' + str(len(path)),
                      ax=ax)

    anim = FuncAnimation(fig, update, frames=total_frames, interval=150)
    plt.close(fig)
    return HTML(anim.to_jshtml())


animate_solution(maze, ROWS, COLS, order, path)

## گام ۷: امتحان با مازهای دیگر

تابع زیر همه‌ی مراحل را برای هر اندازه و هر seed انجام می‌دهد. مقادیر را عوض کنید و دوباره اجرا کنید.

In [ ]:
def run_demo(rows, cols, seed=None):
    m = generate_maze(rows, cols, seed)
    o, p = solve_dfs(m, (0, 0), (rows - 1, cols - 1))
    draw_maze(m, rows, cols, visited=o, path=p,
              title=str(rows) + 'x' + str(cols) + ' | explored ' + str(len(o)) +
                    ' | path ' + str(len(p)))
    plt.show()


run_demo(10, 20, seed=1)
run_demo(25, 25, seed=7)

## تحلیل پیچیدگی

فرض کنید ماز `n` خانه دارد (`n = ROWS × COLS`). ماز کامل یک درخت است، پس تعداد یال‌ها `n - 1` می‌شود.

| بخش | زمان | حافظه |
|---|---|---|
| ساخت ماز با DFS | O(n) | O(n) |
| حل ماز با DFS | O(V + E) = O(n) | O(n) |

- هر خانه حداکثر یک بار ملاقات می‌شود و هر یال حداکثر دو بار بررسی می‌شود.
- بدترین حالت حافظه‌ی پشته وقتی است که مسیر خیلی طولانی و پرپیچ‌وخم باشد.

## نکات مهم

- DFS **کوتاه‌ترین مسیر را تضمین نمی‌کند**. در ماز کامل فقط یک مسیر وجود دارد، پس همان مسیر پیدا می‌شود. اگر ماز چند مسیر داشت، برای کوتاه‌ترین مسیر باید از **BFS** استفاده کرد.
- از پشته‌ی صریح (لیست) استفاده کردیم تا با مازهای بزرگ خطای عمق بازگشت (recursion limit) نگیریم.

## تمرین‌های پیشنهادی

1. الگوریتم BFS را برای حل ماز بنویسید و تعداد خانه‌های بررسی‌شده را با DFS مقایسه کنید.
2. چند دیوار تصادفی را بردارید تا ماز چند مسیر داشته باشد و نتیجه‌ی DFS و BFS را مقایسه کنید.
3. ورودی و خروجی را به مکان‌های دیگر ببرید.